# 02 — Structural Analysis Across Families

This notebook performs structural EDA across all 13 CTU-13 scenarios.
It loads the precomputed topology metrics from `results/metrics/` and
builds the analytical story that motivates the ML approach in notebook 03.

**Prerequisite:** Run `c2detect topology` before this notebook.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from botnet_c2.data.registry import FAMILIES, SCENARIOS, SMALL_SCENARIOS

RESULTS_DIR = Path("../results")
METRICS_DIR = RESULTS_DIR / "metrics"

## 1. Load Topology Metrics

Load all 13 `_topology.json` files and join with registry metadata.

In [ ]:
records = []
for scenario_id, meta in SCENARIOS.items():
    path = METRICS_DIR / f"{scenario_id}_topology.json"
    if not path.exists():
        print(f"WARNING: missing {path}")
        continue
    with open(path) as f:
        m = json.load(f)
    m["family"] = meta["family"]
    m["small"] = scenario_id in SMALL_SCENARIOS
    records.append(m)

metrics_df = pd.DataFrame(records)
print(f"Loaded {len(metrics_df)} scenarios")
metrics_df[
    [
        "scenario_id",
        "family",
        "nodes",
        "edges",
        "density",
        "gc_fraction",
        "max_kcore",
        "powerlaw_gamma",
    ]
].sort_values("family")

## 2. Clustering Coefficient Investigation

**Finding:** `avg_clustering = 0.0` for all 13 scenarios.

This is expected for hub-and-spoke C2 topology. In a star graph, leaf nodes
connect only to the hub — no two leaves share a common neighbor, so no triangles
can form. The local clustering coefficient is therefore 0 by construction for
every node in the graph.

**Implication for the feature set:** `local_clustering` has zero variance
across all scenarios and all nodes, making it useless as an ML feature.
It is excluded from the feature matrix in `features/windows.py`.
`flow_count` (total flows through a node) is used as a proxy for
communication intensity instead.

In [ ]:
print("avg_clustering across all 13 scenarios:")
print(metrics_df[["scenario_id", "family", "avg_clustering"]].to_string(index=False))
print(f"\nAll zero: {(metrics_df['avg_clustering'] == 0.0).all()}")

## 3. Structural Overview Table

Key metrics across all 13 scenarios, sorted by family.
Small scenarios (< 100 node-window observations) are flagged.

In [ ]:
overview_cols = [
    "scenario_id",
    "family",
    "nodes",
    "edges",
    "density",
    "gc_fraction",
    "max_kcore",
    "powerlaw_gamma",
    "robustness_auc_targeted",
    "robustness_auc_random",
    "small",
]
overview = metrics_df[overview_cols].sort_values(["family", "scenario_id"])
overview["powerlaw_gamma"] = overview["powerlaw_gamma"].map(
    lambda x: (
        f"{x:.2f}"
        if x is not None and not (isinstance(x, float) and np.isnan(x))
        else "N/A"
    )
)
overview.style.apply(
    lambda row: ["background-color: #fff3cd" if row["small"] else "" for _ in row],
    axis=1,
)

## 4. Structural Heatmap

Normalized (0-1) structural metrics across all 13 scenarios, sorted by family.
Reveals which families share structural profiles.

In [ ]:
heatmap_cols = [
    "nodes",
    "edges",
    "density",
    "gc_fraction",
    "max_kcore",
    "max_in_degree",
    "max_out_degree",
    "robustness_auc_targeted",
    "robustness_auc_random",
]

hm = metrics_df[["scenario_id", "family"] + heatmap_cols].sort_values(
    ["family", "scenario_id"]
)

# Normalize each column to [0, 1]
hm_norm = hm.copy()
for col in heatmap_cols:
    col_min = hm[col].min()
    col_max = hm[col].max()
    if col_max > col_min:
        hm_norm[col] = (hm[col] - col_min) / (col_max - col_min)
    else:
        hm_norm[col] = 0.0

z = hm_norm[heatmap_cols].values
y_labels = [f"{row['scenario_id']} ({row['family']})" for _, row in hm.iterrows()]

fig = go.Figure(
    go.Heatmap(
        z=z,
        x=heatmap_cols,
        y=y_labels,
        colorscale="Viridis",
        colorbar=dict(title="Normalized value"),
        hoverongaps=False,
    )
)
fig.update_layout(
    title="Structural Metrics Heatmap — 13 Scenarios x 9 Metrics (normalized 0-1)",
    height=550,
    xaxis_tickangle=-30,
    margin=dict(l=200, r=20, t=60, b=80),
)
fig.show()

## 5. Degree Distributions

Log-log CCDF (complementary cumulative degree distribution) for each scenario.
Power-law behavior would appear as a straight line on log-log axes.
Gamma annotations shown where `powerlaw.Fit` succeeded (n >= 50 positive-degree nodes).

In [ ]:
# Family color map
family_colors = {
    fam: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
    for i, fam in enumerate(sorted(FAMILIES))
}

fig = go.Figure()

for _, row in metrics_df.iterrows():
    x = row["degree_ccdf_x"]
    y = row["degree_ccdf_y"]
    if not x or not y:
        continue

    gamma = row["powerlaw_gamma"]
    name = row["scenario_id"]
    if gamma and not np.isnan(gamma):
        name += f" (gamma={gamma:.2f})"

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            name=name,
            line=dict(color=family_colors[row["family"]], width=1.5),
            opacity=0.8,
        )
    )

fig.update_layout(
    title="Degree CCDF — All 13 Scenarios (log-log)",
    xaxis=dict(title="Degree k", type="log"),
    yaxis=dict(title="P(Degree >= k)", type="log"),
    height=500,
    legend=dict(font=dict(size=10)),
)
fig.show()

## 6. Robustness Curves

Giant component fraction as nodes are removed, under targeted (highest-degree first)
vs. random removal. C2 hub-and-spoke graphs are highly vulnerable to targeted
removal — removing the hub immediately fragments the network.

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Targeted removal (highest-degree first)", "Random removal"],
    shared_yaxes=True,
)

for _, row in metrics_df.iterrows():
    targeted = row["robustness_auc_targeted"]
    color = family_colors[row["family"]]
    sid = row["scenario_id"]

    # Reconstruct x-axis from steps count
    steps = 20  # matches _ROBUSTNESS_STEPS in topology.py
    x = list(np.linspace(0, 1, steps))

    t_curve = row.get("robustness_auc_targeted")
    # Note: robustness curves are stored as AUC scalars in topology.json,
    # not as full arrays, so we plot from the JSON directly if arrays available.
    # The topology.json does include full targeted/random lists.

# Reload from JSON to get the full curves
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Targeted removal", "Random removal"],
    shared_yaxes=True,
)

for scenario_id, meta in SCENARIOS.items():
    path = METRICS_DIR / f"{scenario_id}_topology.json"
    if not path.exists():
        continue
    with open(path) as f:
        m = json.load(f)

    color = family_colors[meta["family"]]
    steps = (
        len(m["robustness_auc_targeted"])
        if isinstance(m["robustness_auc_targeted"], list)
        else 20
    )
    x = list(np.linspace(0, 1, steps))

    # targeted and random are stored as full lists in the JSON
    # (auc_targeted and auc_random are scalars; targeted/random are lists)
    for col_idx, key in enumerate(["targeted", "random"], start=1):
        curve = m.get(key, [])
        if not curve:
            continue
        fig.add_trace(
            go.Scatter(
                x=x[: len(curve)],
                y=curve,
                mode="lines",
                name=f"{scenario_id} ({meta['family']})",
                line=dict(color=color, width=1.2),
                opacity=0.7,
                showlegend=(col_idx == 1),
            ),
            row=1,
            col=col_idx,
        )

fig.update_layout(
    title="Network Robustness Curves — Targeted vs. Random Node Removal",
    height=450,
    legend=dict(font=dict(size=9)),
)
fig.update_xaxes(title_text="Fraction of nodes removed")
fig.update_yaxes(title_text="Giant component fraction", col=1)
fig.show()

## 7. Bridge

The structural analysis above confirms that botnet C2 graphs share a consistent
topological signature: low clustering, high in-degree concentration at the hub,
high k-core centrality for the controller, and extreme vulnerability to targeted
removal.

This raises the key ML question: **can a model learn to identify C2 nodes from
these topological features alone, operating on full-traffic graphs without
knowing which flows are botnet in advance?**

**Notebook 03** builds the per-(time_bin, ip) feature matrix that operationalizes
these structural metrics for node-level classification.